# Bloque 4 — Agentes de IA
## Notebook 01: Equipo de tres agentes investigando ContiLeaks

---

### Objetivo

Construir una **crew de tres agentes especializados** que analicen los datos ya procesados de ContiLeaks y produzcan un informe de inteligencia táctica.

Cada agente lee datos distintos y pasa su output al siguiente:

```
┌──────────────────┐     ┌─────────────────────┐     ┌──────────────────┐
│   INVESTIGADOR   │────▶│      ANALISTA        │────▶│    REDACTOR      │
│  qwen3.5:latest  │     │   qwen2.5:14b        │     │ deepseek-r1:14b  │
│                  │     │                      │     │                  │
│ Lee actor_       │     │ Lee conti_sample_    │     │ Recibe outputs   │
│ profiles.json    │     │ classified.parquet   │     │ de los otros dos │
│                  │     │                      │     │                  │
│ Output:          │     │ Output:              │     │ Output:          │
│ Top 5 actores    │     │ Patrones por actor   │     │ Informe TI final │
└──────────────────┘     └─────────────────────┘     └──────────────────┘
```

### Distribución de modelos

| Agente | Modelo | Por qué este modelo |
|---|---|---|
| `investigador` | `qwen3.5:latest` | Tarea de extracción y búsqueda en texto — modelo ágil |
| `analista` | `qwen2.5:14b` | Análisis de patrones y datos estructurados — sólido en razonamiento tabular |
| `redactor` | `deepseek-r1:14b` | Síntesis y redacción de informes — R1 razona internamente (CoT) antes de escribir |

> **Sobre DeepSeek R1**: verás en los logs un bloque `<think>...</think>` que es el razonamiento interno del modelo antes de producir la respuesta final. CrewAI filtra ese bloque del output que pasa al resto de la crew.

### Datos necesarios (en `bloque5_ransomware/data_Vruto/ContiLeaks/`)
- `actor_profiles.json`
- `conti_sample_classified.parquet`

In [1]:
# ─── IMPORTS ─────────────────────────────────────────────────────────────────
import os
import json
import concurrent.futures
from pathlib import Path

import pandas as pd

os.environ['OPENAI_API_KEY'] = 'NA'

from crewai import Agent, Task, Crew, Process, LLM


def kickoff(crew, timeout=600):
    """Ejecuta crew.kickoff() en un hilo separado para evitar conflictos
    con el event loop de Jupyter/nbconvert (problema de CrewAI 1.x)."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(crew.kickoff).result(timeout=timeout)


# ─── RUTAS ───────────────────────────────────────────────────────────────────
DATA = Path('..') / 'bloque5_ransomware' / 'data_Vruto' / 'ContiLeaks'

PROFILES_JSON   = DATA / 'actor_profiles.json'
CLASSIFIED_PARQ = DATA / 'conti_sample_classified.parquet'

assert PROFILES_JSON.exists(),   f'No encontrado: {PROFILES_JSON}'
assert CLASSIFIED_PARQ.exists(), f'No encontrado: {CLASSIFIED_PARQ}'

print('Ficheros de datos localizados correctamente.')

Ficheros de datos localizados correctamente.


In [2]:
# ─── CARGAR DATOS ────────────────────────────────────────────────────────────

# Cargamos los perfiles de actores. Son 30 actores con rol, confianza,
# resumen y ejemplos de mensajes inferidos por el LLM en bloque5.
with open(PROFILES_JSON) as f:
    profiles = json.load(f)

# Cargamos el DataFrame con los mensajes clasificados.
# ~2520 mensajes de los 30 actores más activos, con categoría asignada.
df = pd.read_parquet(CLASSIFIED_PARQ)

print(f'Actores con perfil    : {len(profiles)}')
print(f'Mensajes clasificados : {len(df)}')
print(f'Categorías disponibles: {sorted(df["category"].unique())}')

Actores con perfil    : 30
Mensajes clasificados : 2520
Categorías disponibles: ['comms', 'financial', 'operational', 'organizational', 'technical', 'unknown']


In [3]:
# ─── PREPARAR DATOS COMO TEXTO ───────────────────────────────────────────────
# Los agentes de CrewAI se comunican exclusivamente mediante texto.
# Convertimos nuestros DataFrames y dicts en cadenas que incluiremos
# directamente en las descripciones de las tareas.

# TEXTO 1: Resumen de perfiles para el agente Investigador
lineas_perfiles = []
for actor, info in profiles.items():
    linea = (
        f"- {actor}: rol={info['role']}, confianza={info['confidence']}. "
        f"{info['summary'][:150]}..."
    )
    lineas_perfiles.append(linea)

TEXTO_PERFILES = 'PERFILES DE ACTORES DE CONTI:\n' + '\n'.join(lineas_perfiles)

# TEXTO 2: Distribución de categorías por actor para el agente Analista
# Calculamos cuántos mensajes tiene cada actor en cada categoría.
cats_por_actor = (
    df.groupby(['username', 'category'])
    .size()
    .unstack(fill_value=0)
)

lineas_cats = []
for actor in cats_por_actor.index:
    row = cats_por_actor.loc[actor]
    total = row.sum()
    dominante = row.idxmax()
    lineas_cats.append(
        f"- {actor}: {total} msgs, dominante={dominante} "
        f"({', '.join(f'{c}={v}' for c,v in row.items() if v > 0)})"
    )

TEXTO_CATEGORIAS = 'DISTRIBUCIÓN DE CATEGORÍAS POR ACTOR:\n' + '\n'.join(lineas_cats)

print('Textos de contexto preparados.')
print(f'Perfiles   : {len(TEXTO_PERFILES)} caracteres')
print(f'Categorías : {len(TEXTO_CATEGORIAS)} caracteres')

Textos de contexto preparados.
Perfiles   : 5862 caracteres
Categorías : 3336 caracteres


## Definir los tres LLMs

Configuramos un objeto `LLM` independiente para cada agente. Todos apuntan al mismo Ollama local pero con modelos distintos, lo que nos permite ajustar la capacidad al tipo de tarea.

In [4]:
# ─── CONFIGURACIÓN DE LOS TRES LLMs ──────────────────────────────────────────
OLLAMA_URL = 'http://localhost:11434'

# Extracción de perfiles — necesita velocidad más que razonamiento profundo
llm_investigador = LLM(
    model='ollama/qwen3.5:latest',
    base_url=OLLAMA_URL,
)

# Análisis de patrones y datos tabulares — qwen2.5 destaca en razonamiento estructurado
llm_analista = LLM(
    model='ollama/qwen2.5:14b',
    base_url=OLLAMA_URL,
)

# Síntesis y redacción — deepseek-r1 razona con CoT explícito antes de escribir
llm_redactor = LLM(
    model='ollama/deepseek-r1:14b',
    base_url=OLLAMA_URL,
)

print('LLMs configurados:')
for nombre, llm in [('investigador', llm_investigador), ('analista', llm_analista), ('redactor', llm_redactor)]:
    print(f'  {nombre:15s} → {llm.model}')

LLMs configurados:
  investigador    → qwen3.5:latest
  analista        → qwen2.5:14b
  redactor        → deepseek-r1:14b


## Definir los tres agentes

Cada agente tiene una especialización distinta y un modelo adecuado a su tarea. El `backstory` es tan importante como el `role`: le dice al modelo cómo debe comportarse, qué tono usar y qué limitaciones respetar.

In [5]:
# ─── AGENTE 1: INVESTIGADOR (qwen3.5) ────────────────────────────────────────
# Su trabajo es leer los perfiles de actores e identificar los más relevantes.
# qwen3.5 es suficiente para esta tarea de extracción y priorización.
investigador = Agent(
    role='Investigador de actores de grupos de ransomware',
    goal=(
        'Identificar los actores más influyentes dentro del grupo Conti '
        'analizando sus roles, niveles de confianza y resúmenes de actividad.'
    ),
    backstory=(
        'Eres un investigador de CTI especializado en perfilado de actores. '
        'Has analizado centenares de leaks de grupos criminales. '
        'Tu metodología prioriza el rol funcional sobre la actividad bruta: '
        'un "leader" con pocos mensajes puede ser más relevante que un "operator" prolífico. '
        'Respondes en español con rigor analítico.'
    ),
    llm=llm_investigador,
    verbose=True,
)

# ─── AGENTE 2: ANALISTA (qwen2.5:14b) ────────────────────────────────────────
# Su trabajo es analizar los patrones de comunicación de los actores clave.
# qwen2.5:14b es especialmente bueno razonando sobre tablas y distribuciones.
analista = Agent(
    role='Analista de patrones de comunicación en grupos criminales',
    goal=(
        'Determinar qué tipos de actividad (técnica, operacional, financiera, etc.) '
        'caracterizan a cada actor clave de Conti, basándose en la distribución '
        'de categorías de sus mensajes.'
    ),
    backstory=(
        'Eres un analista especializado en OSINT y análisis de comunicaciones. '
        'Sabes que la categoría dominante en los mensajes de un actor revela su función real: '
        'muchos mensajes "technical" sugieren un developer, muchos "financial" sugieren '
        'un negotiator o manager. Respondes en español con conclusiones concretas.'
    ),
    llm=llm_analista,
    verbose=True,
)

# ─── AGENTE 3: REDACTOR (deepseek-r1:14b) ────────────────────────────────────
# Su trabajo es sintetizar los hallazgos de los dos agentes anteriores.
# deepseek-r1 usa chain-of-thought explícito antes de escribir, lo que produce
# textos más estructurados y argumentados — ideal para un informe de CTI.
redactor = Agent(
    role='Redactor de informes de Threat Intelligence',
    goal=(
        'Producir un informe de inteligencia táctica sobre el grupo Conti '
        'que sintetice el perfilado de actores y los patrones de comunicación '
        'en conclusiones accionables para un equipo de defensa.'
    ),
    backstory=(
        'Eres un redactor técnico con experiencia en informes de threat intelligence '
        'al estilo de PRODAFT, CrowdStrike o Mandiant. '
        'Tus informes son concisos, estructurados y orientados a la toma de decisiones. '
        'Nunca incluyes especulaciones sin base en los datos. '
        'Escribes siempre en español.'
    ),
    llm=llm_redactor,
    verbose=True,
)

print('Tres agentes definidos con modelos distintos.')

Tres agentes definidos con modelos distintos.


## Definir las tres tareas

El parámetro `context` es la clave del trabajo encadenado: le dice a CrewAI que el output de una tarea debe estar disponible como contexto cuando se ejecute la siguiente.

```
tarea_investigacion
        │
        └──▶ tarea_analisis (context=[tarea_investigacion])
                    │
                    └──▶ tarea_informe (context=[tarea_investigacion, tarea_analisis])
```

In [6]:
# ─── TAREA 1: INVESTIGACIÓN DE ACTORES (→ investigador / qwen3.5) ─────────────
# El contexto completo de perfiles se incrusta directamente en la descripción.
tarea_investigacion = Task(
    description=(
        f'Analiza los siguientes perfiles de actores del grupo Conti y selecciona '
        f'los 5 más relevantes para la investigación.\n\n'
        f'{TEXTO_PERFILES}\n\n'
        f'Para cada actor seleccionado, justifica por qué es relevante '
        f'(considera: rol, nivel de confianza, y qué nos dice su resumen de actividad).'
    ),
    expected_output=(
        'Una lista numerada de exactamente 5 actores con el formato:\n'
        '1. [nombre_actor] (rol: X, confianza: Y): [justificación de 1-2 frases]'
    ),
    agent=investigador,
)

# ─── TAREA 2: ANÁLISIS DE PATRONES (→ analista / qwen2.5:14b) ────────────────
# Esta tarea recibe el output de tarea_investigacion como contexto adicional.
# El analista verá tanto los datos de categorías como los 5 actores identificados.
tarea_analisis = Task(
    description=(
        f'Analiza los patrones de comunicación de los actores del grupo Conti.\n\n'
        f'{TEXTO_CATEGORIAS}\n\n'
        f'Foco especial en los actores clave identificados en el análisis anterior. '
        f'Para cada uno de esos actores, determina qué revela su distribución de '
        f'categorías sobre su función real dentro del grupo.'
    ),
    expected_output=(
        'Un análisis por actor (los 5 del análisis anterior) con el formato:\n'
        '- [nombre_actor]: categoría dominante = X (N msgs). '
        'Interpretación: [qué función sugiere esta distribución]'
    ),
    agent=analista,
    context=[tarea_investigacion],  # el output de tarea 1 llega aquí como contexto
)

# ─── TAREA 3: REDACCIÓN DEL INFORME (→ redactor / deepseek-r1:14b) ───────────
# El redactor recibe el output de AMBAS tareas anteriores como contexto.
# deepseek-r1 razonará internamente (bloque <think>) antes de producir el informe.
tarea_informe = Task(
    description=(
        'Redacta un informe de Threat Intelligence sobre el grupo Conti '
        'basándote en los análisis de los agentes anteriores.\n\n'
        'El informe debe incluir:\n'
        '1. Una caracterización del grupo (2-3 frases)\n'
        '2. Los actores clave y sus roles funcionales (lista)\n'
        '3. Patrones de comunicación relevantes para equipos de defensa (párrafo)\n'
        '4. Una conclusión con la hipótesis más sólida sobre la estructura interna del grupo'
    ),
    expected_output=(
        'Un informe estructurado con las cuatro secciones indicadas. '
        'Máximo 400 palabras. Tono profesional, estilo CTI.'
    ),
    agent=redactor,
    context=[tarea_investigacion, tarea_analisis],  # recibe AMBOS análisis
)

print('Tres tareas definidas con dependencias encadenadas.')

Tres tareas definidas con dependencias encadenadas.


In [7]:
# ─── CREAR Y LANZAR EL CREW ───────────────────────────────────────────────────
# Reunimos a los tres agentes y sus tareas en un Crew.
# Process.sequential ejecuta las tareas en orden:
#   investigación (qwen3.5) → análisis (qwen2.5:14b) → redacción (deepseek-r1:14b)
# El contexto fluye automáticamente entre ellas gracias al parámetro `context`.

crew_conti = Crew(
    agents=[investigador, analista, redactor],
    tasks=[tarea_investigacion, tarea_analisis, tarea_informe],
    process=Process.sequential,
    verbose=True,
)

print('Lanzando el crew... (puede tardar 3-6 minutos con tres modelos distintos)')
print('Observa cómo cada agente usa su modelo y razona antes de producir su output.\n')
print('  ① qwen3.5      → identificará los 5 actores clave')
print('  ② qwen2.5:14b  → analizará los patrones de comunicación')
print('  ③ deepseek-r1  → redactará el informe (verás el bloque <think>)\n')

resultado = kickoff(crew_conti)

Lanzando el crew... (puede tardar 3-6 minutos con tres modelos distintos)
Observa cómo cada agente usa su modelo y razona antes de producir su output.

  ① qwen3.5      → identificará los 5 actores clave
  ② qwen2.5:14b  → analizará los patrones de comunicación
  ③ deepseek-r1  → redactará el informe (verás el bloque <think>)



╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.2                                                                                        │
│  Latest version:  1.15.5                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 644dc701-7869-48c2-b908-1509cb366fd7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analiza los siguientes perfiles de actores del grupo Conti y selecciona los 5 más relevantes para la     │
│  investigación.                                                                                                 │
│                                                                                                                 │
│  PERFILES DE ACTORES DE CONTI:                                                                                  │
│  - target: rol=operator, confianza=medium. The individual appears to be an operator involved in the technical   │
│  aspects of ransomware operations, such as setting up servers and managing botnets. ...                         │
│  - bentley: rol=operator, confianza=medium. The individual appears to be an operator involved in the            │
│  distribution and testing of ransomware payloads, managing cryptocurrency wallets for transact...               │
│  - tl1: rol=operator, confianza=medium. This individual appears to be an operator responsible for managing and  │
│  maintaining the infrastructure used in ransomware operations, including handlin...                             │
│  - stern: rol=operator, confianza=medium. The individual appears to be an operator involved in managing the     │
│  ransomware deployment and possibly handling communications with other team members o...                        │
│  - defender: rol=support, confianza=high. This individual appears to be a support member responsible for        │
│  maintaining communication channels and troubleshooting issues within the Conti ransomwa...                     │
│  - hof: rol=operator, confianza=medium. The individual appears to be an operator involved in the day-to-day     │
│  technical operations of deploying and managing bots or malware instances. They com...                          │
│  - user8: rol=operator, confianza=high. The individual is actively involved in the operational aspects of the   │
│  Conti ransomware group, including reconnaissance and lateral movement within a c...                            │
│  - mango: rol=operator, confianza=medium. The individual appears to be an operator within the Conti ransomware  │
│  group, responsible for managing tasks such as testing and deploying malware, hand...                           │
│  - driver: rol=operator, confianza=medium. The individual appears to be an operator responsible for             │
│  maintaining and managing the infrastructure used by the Conti ransomware group. They perform ...               │
│  - deploy: rol=operator, confianza=high. This individual is responsible for deploying ransomware files and      │
│  managing the distribution process. They communicate with other team members about fi...                        │
│  - mushroom: rol=developer, confianza=high. This individual is responsible for coding and testing the           │
│  ransomware software, including debugging issues related to API calls, obfuscation techniques...                │
│  - bio: rol=support, confianza=high. This individual appears to be a support member within the Conti            │
│  ransomware group, likely assisting with technical tasks such as file decryption and da...                      │
│  - baget: rol=operator, confianza=medium. Baget appears to be an operator within the Conti ransomware group,    │
│  responsible for executing tasks and communicating with other members about technica...                         │
│  - ttrr: rol=developer, confianza=high. The individual appears to be a developer working on the technical       │
│  aspects of Conti's infrastructure, specifically dealin

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador de actores de grupos de ransomware                                                         │
│                                                                                                                 │
│  Task: Analiza los siguientes perfiles de actores del grupo Conti y selecciona los 5 más relevantes para la     │
│  investigación.                                                                                                 │
│                                                                                                                 │
│  PERFILES DE ACTORES DE CONTI:                                                                                  │
│  - target: rol=operator, confianza=medium. The individual appears to be an operator involved in the technical   │
│  aspects of ransomware operations, such as setting up servers and managing botnets. ...                         │
│  - bentley: rol=operator, confianza=medium. The individual appears to be an operator involved in the            │
│  distribution and testing of ransomware payloads, managing cryptocurrency wallets for transact...               │
│  - tl1: rol=operator, confianza=medium. This individual appears to be an operator responsible for managing and  │
│  maintaining the infrastructure used in ransomware operations, including handlin...                             │
│  - stern: rol=operator, confianza=medium. The individual appears to be an operator involved in managing the     │
│  ransomware deployment and possibly handling communications with other team members o...                        │
│  - defender: rol=support, confianza=high. This individual appears to be a support member responsible for        │
│  maintaining communication channels and troubleshooting issues within the Conti ransomwa...                     │
│  - hof: rol=operator, confianza=medium. The individual appears to be an operator involved in the day-to-day     │
│  technical operations of deploying and managing bots or malware instances. They com...                          │
│  - user8: rol=operator, confianza=high. The individual is actively involved in the operational aspects of the   │
│  Conti ransomware group, including reconnaissance and lateral movement within a c...                            │
│  - mango: rol=operator, confianza=medium. The individual appears to be an operator within the Conti ransomware  │
│  group, responsible for managing tasks such as testing and deploying malware, hand...                           │
│  - driver: rol=operator, confianza=medium. The individual appears to be an operator responsible for             │
│  maintaining and managing the infrastructure used by the Conti ransomware group. They perform ...               │
│  - deploy: rol=operator, confianza=high. This individual is responsible for deploying ransomware files and      │
│  managing the distribution process. They communicate with other team members about fi...                        │
│  - mushroom: rol=developer, confianza=high. This individual is responsible for coding and testing the           │
│  ransomware software, including debugging issues related to API calls, obfuscation techniques...                │
│  - bio: rol=support, confianza=high. This individual appears to be a support member within the Conti            │
│  ransomware group, likely assisting with technical tasks such as file decryption and da...                      │
│  - baget: rol=operator, confianza=medium. Baget appears to be an operator within the Conti ransomware group,    │
│  responsible for executing tasks and communicating with other members about technica...                         │
│  - ttrr: rol=developer, confianza=high. The individual 

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Redactor de informes de Threat Intelligence                                                             │
│                                                                                                                 │
│  Task: Redacta un informe de Threat Intelligence sobre el grupo Conti basándote en los análisis de los agentes  │
│  anteriores.                                                                                                    │
│                                                                                                                 │
│  El informe debe incluir:                                                                                       │
│  1. Una caracterización del grupo (2-3 frases)                                                                  │
│  2. Los actores clave y sus roles funcionales (lista)                                                           │
│  3. Patrones de comunicación relevantes para equipos de defensa (párrafo)                                       │
│  4. Una conclusión con la hipótesis más sólida sobre la estructura interna del grupo                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Redactor de informes de Threat Intelligence                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Grupo Conti - Análisis Táctico de Threat Intelligence**                                                      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  1. **Caracterización del Grupo**                                                                               │
│  El grupo Conti es un actor cibernético conocido por su participación en actividades de ransomware y robo de    │
│  datos. Su infraestructura técnica incluye la creación de herramientas y servicios customizados, indicando una  │
│  capacidad avanzada en desarrollo malware.                                                                      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  2. **Actores Clave y sus Roles Funcionales**                                                                   │
│  El análisis identifica los siguientes desarrolladores críticos dentro del grupo:                               │
│  - **mushroom**: Desarrollador principal responsable de la creación de API, generación de código malicioso y    │
│  mantenimiento de la carga útil.                                                                                │
│  - **professor**: Experto en bases de datos y shellcodes, con un enfoque en persistencia técnica.               │
│  - **braun**: Especialista en shells de red y enrutamiento, clave para la comunicación y gestión de             │
│  infraestructura.                                                                                               │
│  - **kaktus**: Desarrollador líder en sistemas de carga y especificaciones técnicas, esencial para el diseño    │
│  de herramientas y cargadores.                                                                                  │
│  - **ttrr**: Experto en gestión de servidores y comunicación remota, fundamental para el mantenimiento técnico  │
│  del grupo.                                                                                                     │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  3. **Patrones de Comunicación Relevantes**                                                                     │
│  Los patrones de comunicación dentro del grupo muestran un énfasis en la coordinación técnica y la gestión de   │
│  tareas clave. Los desarrolladores con mayor confianza (mushroom, professor, braun, kaktus, ttrr) lideran las   │
│  discusiones técnicas, mientras los roles mixtos tienden a combinar tareas de desarrollo y comunicación. La     │
│  alta concentración de mensajes técnicos en actores com

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Redacta un informe de Threat Intelligence sobre el grupo Conti basándote en los análisis de los agentes  │
│  anteriores.                                                                                                    │
│                                                                                                                 │
│  El informe debe incluir:                                                                                       │
│  1. Una caracterización del grupo (2-3 frases)                                                                  │
│  2. Los actores clave y sus roles funcionales (lista)                                                           │
│  3. Patrones de comunicación relevantes para equipos de defensa (párrafo)                                       │
│  4. Una conclusión con la hipótesis más sólida sobre la estructura interna del grupo                            │
│  Agent: Redactor de informes de Threat Intelligence                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 644dc701-7869-48c2-b908-1509cb366fd7                                                                       │
│  Final Output: **Grupo Conti - Análisis Táctico de Threat Intelligence**                                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  1. **Caracterización del Grupo**                                                                               │
│  El grupo Conti es un actor cibernético conocido por su participación en actividades de ransomware y robo de    │
│  datos. Su infraestructura técnica incluye la creación de herramientas y servicios customizados, indicando una  │
│  capacidad avanzada en desarrollo malware.                                                                      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  2. **Actores Clave y sus Roles Funcionales**                                                                   │
│  El análisis identifica los siguientes desarrolladores críticos dentro del grupo:                               │
│  - **mushroom**: Desarrollador principal responsable de la creación de API, generación de código malicioso y    │
│  mantenimiento de la carga útil.                                                                                │
│  - **professor**: Experto en bases de datos y shellcodes, con un enfoque en persistencia técnica.               │
│  - **braun**: Especialista en shells de red y enrutamiento, clave para la comunicación y gestión de             │
│  infraestructura.                                                                                               │
│  - **kaktus**: Desarrollador líder en sistemas de carga y especificaciones técnicas, esencial para el diseño    │
│  de herramientas y cargadores.                                                                                  │
│  - **ttrr**: Experto en gestión de servidores y comunicación remota, fundamental para el mantenimiento técnico  │
│  del grupo.                                                                                                     │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  3. **Patrones de Comunicación Relevantes**                                                                     │
│  Los patrones de comunicación dentro del grupo muestran un énfasis en la coordinación técnica y la gestión de   │
│  tareas clave. Los desarrolladores con mayor confianza (mushroom, professor, braun, kaktus, ttrr) lideran las   │
│  discusiones técnicas, mientras los roles mixtos tienden a combinar tareas de desarrollo y comunicación. La     │
│  alta concentración de mensajes técnicos en actores co

In [8]:
# ─── RESULTADO FINAL ──────────────────────────────────────────────────────────
# El resultado del Crew es el output de la ÚLTIMA tarea (el informe del redactor).
# Los outputs intermedios (investigador, analista) sirvieron como contexto
# pero no se devuelven directamente — fluyen internamente entre las tareas.

print('=' * 70)
print('INFORME DE THREAT INTELLIGENCE — GRUPO CONTI')
print('Generado por crew de 3 agentes | qwen3.5 · qwen2.5:14b · deepseek-r1:14b')
print('=' * 70)
print(resultado.raw)

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

INFORME DE THREAT INTELLIGENCE — GRUPO CONTI
Generado por crew de 3 agentes | qwen3.5 · qwen2.5:14b · deepseek-r1:14b
**Grupo Conti - Análisis Táctico de Threat Intelligence**

---

1. **Caracterización del Grupo**  
El grupo Conti es un actor cibernético conocido por su participación en actividades de ransomware y robo de datos. Su infraestructura técnica incluye la creación de herramientas y servicios customizados, indicando una capacidad avanzada en desarrollo malware.

---

2. **Actores Clave y sus Roles Funcionales**  
El análisis identifica los siguientes desarrolladores críticos dentro del grupo:  
- **mushroom**: Desarrollador principal responsable de la creación de API, generación de código malicioso y mantenimiento de la carga útil.  
- **professor**: Experto en bases de datos y shellcodes, con un enfoque en persistencia técnica.  
- **braun**: Especialista en shells de red y enrutamiento, clave para la comunicación y gestión de infraestructura.  
- **kaktus**: Desarrollador 

## Discusión: ¿qué haría falta para un sistema de producción?

Lo que acabamos de construir es un prototipo funcional. Para convertirlo en un sistema real de CTI faltan varias cosas:

### 1. Evaluación
Ahora el output es texto libre. No sabemos si el informe es bueno o malo salvo leyéndolo. Un sistema productivo necesita:
- Una función de scoring automático (otro LLM que evalúe el output)
- Benchmarks con informes de referencia (PRODAFT, CrowdStrike, Mandiant)

### 2. Memoria persistente
Nuestra crew empieza desde cero cada vez. No recuerda lo que analizó ayer. CrewAI soporta memoria persistente con una base de datos vectorial — exactamente lo que aprendimos en bloque3.

### 3. Human-in-the-loop
Algunos sistemas de CTI requieren que un analista humano apruebe los hallazgos antes de que el redactor los incluya en el informe. CrewAI soporta `human_input=True` en las tareas críticas.

### 4. Selección dinámica de modelo
En vez de asignar un modelo fijo por agente, un sistema avanzado podría elegir el modelo en función de la complejidad estimada de cada tarea — usando modelos más pequeños para tareas simples y reservando los grandes para las complejas.

---

**Siguiente**: en `02_agentes_con_herramientas.ipynb` añadimos **herramientas** al agente — funciones Python que el LLM puede invocar para consultar los datos con precisión en vez de recibirlos todos en el contexto.